# Distances Between Observations

Read this notebook from top to bottom and fill in the code as you go. Work together and discuss with other students in the class. Try to resolve any errors on your own first, but don't get stuck; ask for help!

In addition to writing and running code, be sure to examine any output and interpret the results before moving on.

For many of these questions, there are several approaches, and there is no single right answer. You should try a few different things and compare with your classmates.

We will use `scikit-learn` extensively later, but for this activity you might want to stick with `pandas`.


In [1]:
import pandas as pd
import numpy as np

## Ames - Recommending Similar Homes

1\. Suppose that you really like house 0 in the Ames housing data set, but it is too expensive. Find cheaper homes that are similar to it --- in terms of living area, number of bedrooms, number of bathrooms --- by calculating distances from house 0. You might want to try different distance metrics and different scaling methods; how sensitive are your results to these choices?

Be sure to actually look at the profiles of the homes that your algorithm picked out as most similar (based on these 3 variables). Do they make sense?

**_Think:_ If the goal is to find a "good deal" on a similar house, should sale price be included as a variable in your distance metric?**

In [2]:
df_ames = pd.read_csv("https://datasci112.stanford.edu/data/housing.tsv", sep="\t")
df_ames["Bathrooms"] = df_ames["Full Bath"] + 0.5 * df_ames["Half Bath"]

def scale_numeric(frame, method):
    if method == "raw":
        return frame.astype(float)
    if method == "standard":
        return (frame - frame.mean()) / frame.std(ddof=0)
    if method == "minmax":
        return (frame - frame.min()) / (frame.max() - frame.min())
    raise ValueError("Unknown scaling method")

def numeric_distances(frame, columns, metric="euclidean", scale="standard"):
    values = scale_numeric(frame[columns], scale)
    differences = values.subtract(values.iloc[0])
    if metric == "euclidean":
        distance = np.sqrt((differences ** 2).sum(axis=1))
    elif metric == "manhattan":
        distance = differences.abs().sum(axis=1)
    else:
        raise ValueError("Unknown distance metric")
    return frame.assign(distance=distance)

house_features = ["Gr Liv Area", "Bedroom AbvGr", "Bathrooms"]
house0_profile = df_ames.loc[[0], ["PID", *house_features, "House Style", "SalePrice"]]
house0_price = df_ames.loc[0, "SalePrice"]

part1_tables = {}
for scale in ["raw", "standard", "minmax"]:
    distances = numeric_distances(df_ames, house_features, scale=scale)
    candidates = distances.loc[(distances.index != 0) & (distances["SalePrice"] < house0_price)]
    part1_tables[f"{scale} Euclidean"] = candidates.sort_values("distance").head(5)

standard_manhattan = numeric_distances(df_ames, house_features, metric="manhattan", scale="standard")
standard_manhattan = standard_manhattan.loc[(standard_manhattan.index != 0) & (standard_manhattan["SalePrice"] < house0_price)]
part1_tables["standard Manhattan"] = standard_manhattan.sort_values("distance").head(5)

part1_results = pd.concat(part1_tables, names=["method", "row"])[["PID", *house_features, "House Style", "SalePrice", "distance"]]
display(house0_profile)
part1_results

,PID,Gr Liv Area,Bedroom AbvGr,Bathrooms,House Style,SalePrice
0,526301100,1656,3,1.0,1Story,215000


PID  Gr Liv Area  Bedroom AbvGr  Bathrooms  \
method             row                                                      
raw Euclidean      1550  910205010         1656              3        1.5   
                   1197  534250010         1656              4        2.0   
                   1293  902109120         1656              2        2.0   
                   2638  902103090         1657              4        1.0   
                   1927  535181030         1657              3        2.0   
standard Euclidean 1226  534477270         1661              3        1.0   
                   1940  535353050         1647              3        1.0   
                   291   909250030         1666              3        1.0   
                   758   903481120         1666              3        1.0   
                   1357  903427090         1666              3        1.0   
minmax Euclidean   1226  534477270         1661              3        1.0   
                   1940  535353050         1647              3        1.0   
                   758   903481120         1666              3        1.0   
                   291   909250030         1666              3        1.0   
                   1357  903427090         1666              3        1.0   
standard Manhattan 1226  534477270         1661              3        1.0   
                   1940  535353050         1647              3        1.0   
                   1357  903427090         1666              3        1.0   
                   758   903481120         1666              3        1.0   
                   291   909250030         1666              3        1.0   

                        House Style  SalePrice  distance  
method             row                                    
raw Euclidean      1550        SLvl     126000  0.500000  
                   1197      1Story     135000  1.414214  
                   1293      1.5Fin     119164  1.414214  
                   2638      1.5Fin     111500  1.414214  
                   1927      1Story     163500  1.414214  
standard Euclidean 1226        SLvl     165500  0.009893  
                   1940      1Story     153000  0.017807  
                   291       1.5Fin     100000  0.019785  
                   758       1.5Fin     135000  0.019785  
                   1357      2Story     161000  0.019785  
minmax Euclidean   1226        SLvl     165500  0.000942  
                   1940      1Story     153000  0.001696  
                   758       1.5Fin     135000  0.001884  
                   291       1.5Fin     100000  0.001884  
                   1357      2Story     161000  0.001884  
standard Manhattan 1226        SLvl     165500  0.009893  
                   1940      1Story     153000  0.017807  
                   1357      2Story     161000  0.019785  
                   758       1.5Fin     135000  0.019785  
                   291       1.5Fin     100000  0.019785

House 0 costs $215,000. It has 1,656 square feet of living area, 3 bedrooms, and 1.0 bathrooms when a half bath counts as 0.5.

I used z scored Euclidean distance for the main comparison. Standardizing prevents living area from overwhelming bedrooms and bathrooms. The nearest cheaper homes have living areas near 1,656 square feet and usually have the same bedroom and bathroom counts. Raw distance favors exact living-area matches because square feet has a much larger numeric scale. Min-max scaling gives nearly the same ranking as z score scaling. Standardized Manhattan and Euclidean distance also produce nearly the same top homes.

I did not include SalePrice in the distance. I used it as a filter after measuring similarity. Including price would favor cheap homes instead of finding homes that match House 0 on its physical characteristics.

2\. Continuing part 1. Suppose that you really like house 0 in the data set, but it is too expensive. Find cheaper homes that are similar to it --- in terms of living area, number of bedrooms, number of bathrooms, **and House Style** --- by calculating distances from house 0. You might want to try different distance metrics and different scaling methods; how sensitive are your results to these choices?

Be sure to actually look at the profiles of the homes that your algorithm picked out as most similar. Do they make sense?

In [3]:
def mixed_distances(frame, numeric_columns, categorical_columns, scale="standard"):
    numeric_values = scale_numeric(frame[numeric_columns], scale)
    categorical_values = pd.get_dummies(
        frame[categorical_columns].fillna("Missing"),
        columns=categorical_columns,
        dtype=float,
    )
    values = pd.concat([numeric_values, categorical_values], axis=1)
    differences = values.subtract(values.iloc[0])
    distance = np.sqrt((differences ** 2).sum(axis=1))
    return frame.assign(distance=distance)

part2_tables = {}
for scale in ["raw", "standard", "minmax"]:
    distances = mixed_distances(df_ames, house_features, ["House Style"], scale=scale)
    candidates = distances.loc[(distances.index != 0) & (distances["SalePrice"] < house0_price)]
    part2_tables[f"{scale} numeric + House Style"] = candidates.sort_values("distance").head(5)

part2_results = pd.concat(part2_tables, names=["method", "row"])[["PID", *house_features, "House Style", "SalePrice", "distance"]]
part2_results

PID  Gr Liv Area  Bedroom AbvGr  \
method                         row                                           
raw numeric + House Style      1927  535181030         1657              3   
                               1197  534250010         1656              4   
                               1550  910205010         1656              3   
                               2638  902103090         1657              4   
                               1293  902109120         1656              2   
standard numeric + House Style 1940  535353050         1647              3   
                               618   534476150         1644              3   
                               2700  904100170         1640              3   
                               314   916125360         1687              3   
                               788   905450020         1689              3   
minmax numeric + House Style   1940  535353050         1647              3   
                               618   534476150         1644              3   
                               2700  904100170         1640              3   
                               314   916125360         1687              3   
                               788   905450020         1689              3   

                                     Bathrooms House Style  SalePrice  \
method                         row                                      
raw numeric + House Style      1927        2.0      1Story     163500   
                               1197        2.0      1Story     135000   
                               1550        1.5        SLvl     126000   
                               2638        1.0      1.5Fin     111500   
                               1293        2.0      1.5Fin     119164   
standard numeric + House Style 1940        1.0      1Story     153000   
                               618         1.0      1Story     167000   
                               2700        1.0      1Story     131000   
                               314         1.0      1Story     160000   
                               788         1.0      1Story     127500   
minmax numeric + House Style   1940        1.0      1Story     153000   
                               618         1.0      1Story     167000   
                               2700        1.0      1Story     131000   
                               314         1.0      1Story     160000   
                               788         1.0      1Story     127500   

                                     distance  
method                         row             
raw numeric + House Style      1927  1.414214  
                               1197  1.414214  
                               1550  1.500000  
                               2638  2.000000  
                               1293  2.000000  
standard numeric + House Style 1940  0.017807  
                               618   0.023743  
                               2700  0.031657  
                               314   0.061335  
                               788   0.065292  
minmax numeric + House Style   1940  0.001696  
                               618   0.002261  
                               2700  0.003014  
                               314   0.005840  
                               788   0.006217

I encoded House Style with one hot columns and kept the quantitative variables on a standard scale. The closest cheaper homes use the 1Story style, which matches House 0. The top profiles also have living areas near 1,656 square feet, 3 bedrooms, and 1 bathroom.

The ranking changes when the quantitative variables stay on their raw scales because living area then dominates the distance. Standard scaling gives House Style a meaningful contribution and produces a more balanced comparison.

3\. Continuing parts 1 and 2. Suppose that you really like house 0 in the data set, but it is too expensive. Find cheaper homes that are similar to it, by calculating distances. You can **choose the variables to include, but include both quantitative and categorical variables**. Be sure to actually look at the profiles of the homes that your algorithm picked out as most similar. Do they make sense?

You might want to try different distance metrics and different scaling methods; how sensitive are your results to these choices?

_Hint:_ There are many variables in the data set. Do not attempt to compute distance based on all the variables! You will want to pare down the number of variables, but be sure to include a mixture of categorical and quantitative variables. Refer to the [data documentation](https://ww2.amstat.org/publications/jse/v19n3/decock/DataDocumentation.txt) for information about the variables.


In [4]:
part3_numeric = ["Gr Liv Area", "Bedroom AbvGr", "Bathrooms", "Overall Qual", "Year Built"]
part3_categorical = ["Neighborhood", "House Style", "Central Air"]
part3_distances = mixed_distances(
    df_ames,
    part3_numeric,
    part3_categorical,
    scale="standard",
)
part3_candidates = part3_distances.loc[(part3_distances.index != 0) & (part3_distances["SalePrice"] < house0_price)]
part3_results = part3_candidates.sort_values("distance").head(10)
part3_results[["PID", *part3_numeric, *part3_categorical, "SalePrice", "distance"]]

,PID,Gr Liv Area,Bedroom AbvGr,Bathrooms,Overall Qual,Year Built,Neighborhood,House Style,Central Air,SalePrice,distance
1240,535176100,1570,3,1.0,6,1958,NAmes,1Story,Y,166800,0.182556
618,534476150,1644,3,1.0,6,1953,NAmes,1Story,Y,167000,0.232694
1896,534425080,1429,3,1.0,6,1960,NAmes,1Story,Y,181900,0.449129
989,526351030,1414,3,1.0,6,1958,NAmes,1Story,Y,176500,0.483353
1894,534403420,1374,3,1.0,6,1964,NAmes,1Story,Y,147000,0.573414
1895,534425015,1652,3,1.5,6,1959,NAmes,1Story,Y,200000,0.778636
1239,535153150,1261,3,1.0,6,1958,NAmes,1Story,Y,163000,0.784318
147,535179020,1580,3,1.5,6,1959,NAmes,1Story,Y,159500,0.792983
1216,534427040,1252,3,1.0,6,1959,NAmes,1Story,Y,142000,0.800015
2545,534403280,1251,3,1.0,6,1964,NAmes,1Story,Y,160000,0.812154


I used living area, bedrooms, bathrooms, overall quality, and year built as quantitative variables. I used neighborhood, House Style, and Central Air as categorical variables. I standardized the quantitative columns and one hot encoded the categorical columns.

The closest cheaper homes are in NAmes, use the 1Story style, have central air, and have an overall quality score of 6. These matches make sense because they share several important characteristics with House 0. SalePrice stays outside the distance calculation, so the results still identify potential lower priced deals.

## Colleges similar to Cal Poly

We'll use data from the [College Scorecard data](https://collegescorecard.ed.gov/) to find colleges and universities that are similar to Cal Poly.

In [5]:
df_college = pd.read_csv("https://datasci112.stanford.edu/data/college_attributes.csv")

df_college.set_index("Institution", inplace = True)

df_college

,City,State,AdmissionRate,Undergraduates,CarnegieClassification,Ownership,PCIP01,PCIP03,PCIP04,PCIP05,...,PCIP44,PCIP45,PCIP46,PCIP47,PCIP48,PCIP49,PCIP50,PCIP51,PCIP52,PCIP54
Institution,,,,,,,,,,,,,,,,,,,,,
Alabama A & M University,Normal,AL,0.7160,5098.0,Master's Colleges & Universities: Larger Programs,Public,0.0445,0.0071,0.0053,0.0000,...,0.0409,0.0249,0.0,0.0,0.0,0.0,0.0231,0.0000,0.1637,0.0000
University of Alabama at Birmingham,Birmingham,AL,0.8854,13284.0,Doctoral Universities: Very High Research Acti...,Public,0.0000,0.0000,0.0000,0.0020,...,0.0195,0.0239,0.0,0.0,0.0,0.0,0.0249,0.2088,0.2159,0.0141
University of Alabama in Huntsville,Huntsville,AL,0.7367,7358.0,Doctoral Universities: Very High Research Acti...,Public,0.0000,0.0000,0.0000,0.0000,...,0.0000,0.0127,0.0,0.0,0.0,0.0,0.0407,0.1341,0.1930,0.0073
Alabama State University,Montgomery,AL,0.9799,3495.0,Doctoral/Professional Universities,Public,0.0000,0.0000,0.0000,0.0000,...,0.0648,0.0196,0.0,0.0,0.0,0.0,0.0511,0.0904,0.1513,0.0059
The University of Alabama,Tuscaloosa,AL,0.7890,30725.0,Doctoral Universities: Very High Research Acti...,Public,0.0000,0.0061,0.0000,0.0019,...,0.0072,0.0661,0.0,0.0,0.0,0.0,0.0234,0.1077,0.2916,0.0096
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Florida Academy of Nursing,Miramar,FL,0.3088,239.0,Not applicable,Private for-profit,0.0000,0.0000,0.0000,0.0000,...,0.0000,0.0000,0.0,0.0,0.0,0.0,0.0000,1.0000,0.0000,0.0000
Herzing University-Tampa,Tampa,FL,0.9630,68.0,Not applicable,Private nonprofit,0.0000,0.0000,0.0000,0.0000,...,0.0000,0.0000,0.0,0.0,0.0,0.0,0.0000,0.0000,0.0000,0.0000
Abilene Christian University-Undergraduate Online,Addison,TX,1.0000,415.0,Not applicable,Private nonprofit,0.0000,0.0000,0.0000,0.0000,...,0.0000,0.0000,0.0,0.0,0.0,0.0,0.0000,0.0000,0.0000,0.0000


We'll want to single out Cal Poly, which we can do like this.

In [6]:
school_name = "California Polytechnic State University-San Luis Obispo"

cp = df_college.loc[school_name]

cp

City                                                        San Luis Obispo
State                                                                    CA
AdmissionRate                                                          0.33
Undergraduates                                                      21090.0
CarnegieClassification    Master's Colleges & Universities: Larger Programs
Ownership                                                            Public
PCIP01                                                               0.1084
PCIP03                                                               0.0255
PCIP04                                                               0.0441
PCIP05                                                               0.0019
PCIP09                                                               0.0353
PCIP10                                                               0.0175
PCIP11                                                               0.0326
PCIP12      

1\. Based on only the admission rate and the number of undergraduates, what schools are most similar to Cal Poly? Specify how you're making this decision.

In [7]:
def college_distances(numeric_columns, categorical_columns=None):
    categorical_columns = [] if categorical_columns is None else list(categorical_columns)
    numeric_values = scale_numeric(df_college[numeric_columns], "standard")
    if categorical_columns:
        categorical_values = pd.get_dummies(
            df_college[categorical_columns].fillna("Missing"),
            columns=categorical_columns,
            dtype=float,
        )
    else:
        categorical_values = pd.DataFrame(index=df_college.index)
    values = pd.concat([numeric_values, categorical_values], axis=1)
    differences = values.subtract(values.loc[school_name])
    distance = np.sqrt((differences ** 2).sum(axis=1))
    return df_college.assign(distance=distance).drop(index=school_name).sort_values("distance")

college_part1 = college_distances(["AdmissionRate", "Undergraduates"])
college_part1[["City", "State", "AdmissionRate", "Undergraduates", "distance"]].head(10)

,City,State,AdmissionRate,Undergraduates,distance
Institution,,,,,
University of California-Santa Barbara,Santa Barbara,CA,0.2918,23081.0,0.309241
DeVry University-Illinois,Naperville,IL,0.4552,19729.0,0.593272
University of North Carolina at Chapel Hill,Chapel Hill,NC,0.2040,19722.0,0.596999
Clemson University,Clemson,SC,0.4922,21577.0,0.736976
University of Virginia-Main Campus,Charlottesville,VA,0.2074,17041.0,0.761491
CUNY Hunter College,New York,NY,0.4590,17293.0,0.761636
Stony Brook University,Stony Brook,NY,0.4806,17900.0,0.795959
Boston University,Boston,MA,0.1865,17501.0,0.797245
North Carolina State University at Raleigh,Raleigh,NC,0.4748,24999.0,0.826475


I standardized admission rate and undergraduate enrollment, then ranked schools by Euclidean distance from Cal Poly. This gives both variables equal influence despite their different units. UC Santa Barbara is the closest match. Its admission rate is 0.2918 and it has 23,081 undergraduates, compared with Cal Poly's 0.33 and 21,090.

2\. Now consider the admission rate, the number of undergraduates, and also the [Carnegie classification](https://en.wikipedia.org/wiki/Carnegie_Classification_of_Institutions_of_Higher_Education) of the type of school, and the ownership (public, private, etc.) Based on these variables, what schools are most similar to Cal Poly? Specify how you're making this decision.

In [8]:
college_part2 = college_distances(
    ["AdmissionRate", "Undergraduates"],
    ["CarnegieClassification", "Ownership"],
)
college_part2[["City", "State", "AdmissionRate", "Undergraduates", "CarnegieClassification", "Ownership", "distance"]].head(10)

,City,State,AdmissionRate,Undergraduates,CarnegieClassification,Ownership,distance
Institution,,,,,,,
CUNY Hunter College,New York,NY,0.4590,17293.0,Master's Colleges & Universities: Larger Programs,Public,0.761636
CUNY Bernard M Baruch College,New York,NY,0.5056,15483.0,Master's Colleges & Universities: Larger Programs,Public,1.073875
CUNY John Jay College of Criminal Justice,New York,NY,0.4458,12834.0,Master's Colleges & Universities: Larger Programs,Public,1.185292
CUNY Brooklyn College,Brooklyn,NY,0.5136,12567.0,Master's Colleges & Universities: Larger Programs,Public,1.376673
University of California-Santa Barbara,Santa Barbara,CA,0.2918,23081.0,Doctoral Universities: Very High Research Acti...,Public,1.447629
California State Polytechnic University-Pomona,Pomona,CA,0.6062,26802.0,Master's Colleges & Universities: Larger Programs,Public,1.450667
CUNY Queens College,Queens,NY,0.6078,14859.0,Master's Colleges & Universities: Larger Programs,Public,1.491767
DeVry University-Illinois,Naperville,IL,0.4552,19729.0,Master's Colleges & Universities: Larger Programs,Private for-profit,1.533614
University of North Carolina at Chapel Hill,Chapel Hill,NC,0.2040,19722.0,Doctoral Universities: Very High Research Acti...,Public,1.535059


I kept the standardized numeric distance and added one-hot columns for Carnegie classification and ownership. CUNY Hunter College is the closest match. It shares Cal Poly's public ownership and the Carnegie classification "Master's Colleges & Universities: Larger Programs." The categorical match moves it above schools that are closer on size and admission rate but have a different classification.

3\. The columns whose names begin with "PCIP" contain the proportions of students at each school studying various fields (e.g., Engineering, Psychology). Each field is represented by a two-digit code called the [CIP code](https://nces.ed.gov/ipeds/cipcode/browse.aspx?y=55).

If we only consider the proportions of students studying various fields, what schools are most similar to Cal Poly? Specify how you're making this decision.

In [9]:
pcip_columns = df_college.filter(regex="^PCIP").columns.tolist()
college_part3 = college_distances(pcip_columns)
college_part3[["City", "State", "distance"]].head(10)

,City,State,distance
Institution,,,
Iowa State University,Ames,IA,1.986715
California State Polytechnic University-Pomona,Pomona,CA,2.242299
Texas A & M University-College Station,College Station,TX,2.259519
Mississippi State University,Mississippi State,MS,2.335885
Clemson University,Clemson,SC,2.377477
North Carolina State University at Raleigh,Raleigh,NC,2.509641
West Virginia University,Morgantown,WV,2.584855
Louisiana Tech University,Ruston,LA,2.779416
Louisiana State University and Agricultural & Mechanical College,Baton Rouge,LA,2.803197


I selected all 38 PCIP columns, standardized each field proportion, and used Euclidean distance. Iowa State University is the closest match, followed by Cal Poly Pomona and Texas A&M University. These schools have similar mixes of students across academic fields, with engineering and business making large contributions to the distance comparison.